# Generador de Demanda Vehicular Estocastica para SUMO

Basado en datos de la Tabla 14 - Tesis E. Sarango (2025)
Interseccion: Av. Isidro Ayora / Av. 8 de Diciembre - Loja

Red: Simple_Intersection.net.xml
  Entradas: -E5 (Norte->J11), -E4 (Oeste->J11)
  Salidas:  E3 (J11->Este),   E6 (J11->Sur)

Movimientos permitidos (segun connections del .net.xml):
  -E5 -> E6  (Norte->Sur,   recto)
  -E5 -> E3  (Norte->Este,  giro izquierda)
  -E4 -> E3  (Oeste->Este,  recto)
  -E4 -> E6  (Oeste->Sur,   giro derecha)

In [1]:
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom

## Parametros Generales

Referencia:
- SEED=42: garantiza reproducibilidad de resultados
- DURACION=3600s (1 hora): periodo de simulacion
- INTERVALO=60s: muestreo Poisson cada minuto
- LAMBDA_MIN=19.4771 veh/min: tasa promedio observada en campo
  (calculada como promedio de 7 muestras de video, Tabla 13 Sarango 2025)

In [2]:
SEED       = 42
DURACION   = 3600    # segundos
INTERVALO  = 60      # muestreo Poisson cada 60s
LAMBDA_MIN = 19.4771 # lambda por acceso (veh/min)

PERIODOS = [
    (0, 3600, 1.0, "Hora pico"),  # lambda = 19.48 veh/min constante
]

## Rutas por Acceso

Referencia:
- Acceso Oeste (-E4): Av. Isidro Ayora (via principal)
- Acceso Norte (-E5): Av. 8 de Diciembre (via secundaria)
- Probabilidades de giro: 70% recto, 30% giro (estimacion ingenieria de trafico)

In [3]:
ACCESOS = [
    {
        "nombre": "Oeste",
        "rutas": [
            ("ruta_OE", "-E4", "E3", 0.70),  # Oeste->Este (recto)
            ("ruta_OS", "-E4", "E6", 0.30),  # Oeste->Sur  (giro derecha)
        ]
    },
    {
        "nombre": "Norte",
        "rutas": [
            ("ruta_NS", "-E5", "E6", 0.70),  # Norte->Sur  (recto)
            ("ruta_NE", "-E5", "E3", 0.30),  # Norte->Este (giro izquierda)
        ]
    },
]

## Composicion Vehicular

Referencia:
- Conteo realizado por YOLOv5 en video de campo (Sarango 2025)
- Ligeros: 7032 (69.58%)
- Motos: 885 (8.76%)
- Pesados/Buses: 2189 (21.66%)

In [4]:
conteo_car  = 7032
conteo_moto = 885
conteo_bus  = 2189

total_tipos = conteo_car + conteo_moto + conteo_bus

TIPOS = [
    ("car",  "passenger",  round(conteo_car  / total_tipos, 4)),
    ("moto", "motorcycle", round(conteo_moto / total_tipos, 4)),
    ("bus",  "bus",        round(conteo_bus  / total_tipos, 4)),
]

## Generacion con Curva Weibull

Referencia:
- Distribucion Weibull modela la forma asimetrica del trafico durante la hora pico
- PDF: f(x) = (k/c) * (x/c)^(k-1) * exp(-(x/c)^k)
- shape (k) = 2.5: controla la asimetria (picos mas pronunciados)
- scale (c) = 0.55: centra el pico hacia el 55% de la duracion

Metodo:
1. Calcular tasa dinamica Weibull para cada intervalo de 60s
2. Aplicar Poisson sobre esa tasa para mantener estocasticidad micro
3. Distribuir vehiculos aleatoriamente dentro de cada intervalo

In [5]:
def lambda_weibull(t, total_vehiculos, duracion=3600, shape=2.5, scale=0.55):
    x = t / duracion
    if x <= 0: return 0.0
    pdf = (shape / scale) * ((x / scale) ** (shape - 1)) * np.exp(- (x / scale) ** shape)
    vehiculos_en_este_intervalo = total_vehiculos * pdf * (INTERVALO / duracion)
    return vehiculos_en_este_intervalo

def generar_tiempos_acceso(seed_offset, total_esperado=1167):
    rng = np.random.default_rng(SEED + seed_offset)
    tiempos = []
    for t0 in range(0, DURACION, INTERVALO):
        tasa_dinamica = lambda_weibull(t0, total_esperado)
        n = rng.poisson(tasa_dinamica)
        offsets = rng.uniform(0, INTERVALO, size=n)
        tiempos.extend(t0 + offsets)
    tiempos.sort()
    return tiempos

## Construccion del XML

Genera el archivo `.rou.xml` compatible con SUMO.

Componentes:
- vTypes: definiciones de vehiculos (car, moto, bus) con modelo Krauss
- routes: rutas permitidas por cada acceso
- vehicles: lista de vehiculos con tiempo de llegada, tipo y ruta

In [6]:
def construir_xml(vehiculos_por_acceso):
    root = ET.Element("routes")
    root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root.set("xsi:noNamespaceSchemaLocation", "http://sumo.dlr.de/xsd/routes_file.xsd")

    ET.SubElement(root, "vType", id="car",  vClass="passenger",
                  accel="3.0", decel="4.5", sigma="0.5",
                  length="5.0", minGap="2.5", maxSpeed="13.9",
                  carFollowModel="Krauss", color="0.8,0.8,0.8")
    ET.SubElement(root, "vType", id="moto", vClass="motorcycle",
                  accel="3.5", decel="5.0", sigma="0.6",
                  length="2.2", minGap="2.0", maxSpeed="16.7",
                  carFollowModel="Krauss", color="0.9,0.5,0.1")
    ET.SubElement(root, "vType", id="bus",  vClass="bus",
                  accel="2.0", decel="4.5", sigma="0.3",
                  length="12.0", minGap="2.5", maxSpeed="11.1",
                  carFollowModel="Krauss", color="0.2,0.6,0.2")

    for acceso in ACCESOS:
        for rid, e_from, e_to, _ in acceso["rutas"]:
            ET.SubElement(root, "route", id=rid, edges=f"{e_from} {e_to}")

    todos = []
    for acceso, (tiempos, rng_seed) in zip(ACCESOS, vehiculos_por_acceso):
        rng = np.random.default_rng(rng_seed)
        rutas  = acceso["rutas"]
        p_r    = np.array([r[3] for r in rutas]); p_r /= p_r.sum()
        p_t    = np.array([v[2] for v in TIPOS]);  p_t /= p_t.sum()
        idx_r  = rng.choice(len(rutas), size=len(tiempos), p=p_r)
        idx_t  = rng.choice(len(TIPOS), size=len(tiempos), p=p_t)
        for t, ir, it in zip(tiempos, idx_r, idx_t):
            todos.append({"depart": t, "type": TIPOS[it][0], "route": rutas[ir][0]})
    todos.sort(key=lambda x: x["depart"])

    for i, v in enumerate(todos):
        ET.SubElement(root, "vehicle", id=f"veh_{i}", type=v["type"],
                      route=v["route"], depart=f"{v['depart']:.2f}",
                      departLane="best", departSpeed="0")
    return root, len(todos)

def guardar(root, filepath):
    xml_str = ET.tostring(root, encoding='utf-8')
    parsed_str = minidom.parseString(xml_str)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(parsed_str.toprettyxml(indent="    "))

## Generar Archivo de Rutas

Genera `loja_intersection_weibull.rou.xml` con la demanda vehicular.
- 1205 vehiculos por acceso (total 2410 en 1 hora)
- Seed diferente por acceso para independencia estadistica

In [7]:
tiempos_oeste = generar_tiempos_acceso(seed_offset=0, total_esperado=1205)
tiempos_norte = generar_tiempos_acceso(seed_offset=10, total_esperado=1205)

vehiculos_por_acceso = [
    (tiempos_oeste, SEED + 1),
    (tiempos_norte, SEED + 11),
]

root, total = construir_xml(vehiculos_por_acceso)
out = r"./loja_intersection_weibull.rou.xml"
guardar(root, out)
print(f"Total de vehiculos inyectados: {total}")
print(f"Archivo listo: {out}")

Total de vehiculos inyectados: 2418
Archivo listo: ./loja_intersection_weibull.rou.xml
